# Settings


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sympy as sp
import scipy as sc
import astropy.cosmology as ac
import astropy.constants as cst
from matplotlib.patches import Ellipse
from matplotlib.gridspec import GridSpec
import sys
from scipy.stats import norm
import statistics as stat
import scipy.stats as stats
from matplotlib.patches import Patch


c:\Users\cleam\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\cleam\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [2]:
!ls

__pycache__
animations
basket notebook.ipynb
basket.py
data
distribution2.png
fisher_matrix_analysis.py
fisher_matrix_analysis_notebook.ipynb
pictures
results


In [3]:
import fisher_matrix_analysis as fma

In [4]:
# fiducial parameters for fLCDM LSST data

H0=70
Mb=-19.3 

Om0_fCPL= 0.315
Ode0_fCPL=1-Om0_fCPL
w0_fCPL = -1.029
wa_fCPL= 0.21

fiducial_fCPL=[Om0_fCPL,w0_fCPL,wa_fCPL,Mb]

# other parameters
c=float(np.array(cst.c.to('km/s')))
 
# fiducial cosmology
cosmo_fCPL=ac.Flatw0waCDM(H0=H0,Om0=Om0_fCPL,w0=w0_fCPL,wa=wa_fCPL)

In [5]:
# load data
LSST=np.load('data/LSST_fCPL.npz') #fiducial cosmology : kPEDE, binned data, no Gaussian noise added
z=LSST['z']
mb=LSST['mb']
sigma_stat=LSST['sigma_stat']

# covariance matrix

# statistical error
rho = 0.4
std_matrix =  np.diag(sigma_stat)
corr_matrix = np.full((len(sigma_stat),len(sigma_stat)),rho)
np.fill_diagonal(corr_matrix,1.0)
Cov_stat_lsst = std_matrix @ corr_matrix @ std_matrix

# systematic error
Cov_sys_lsst=np.array(pd.read_table('data/covsys_lsst.csv',sep=',')).reshape((len(z),len(z))) 

# total covariance matrix
Cov_tot_lsst= Cov_sys_lsst + Cov_stat_lsst
Cov_inv_lsst=np.linalg.inv(Cov_tot_lsst)

# Fisher Matrix Analysis

In [6]:
data_obs = mb #observed data

In [7]:
fma.chi2(fiducial_fCPL,z,data_obs,Cov_tot_lsst,'fCPL',H0)

21.95109644215517

In [8]:
F=fma.fisher_matrix(fma.chi2, fiducial_fCPL, 'fCPL', H0, z, data_obs, Cov_tot_lsst)

In [9]:
param_names=['Om0','w0','wa','Mb']
param_labels=[r'$\Omega_m$',r'$w_0$',r'$w_a$',r'$M_B$']
C_w0_wa=fma.marginalize_fisher_matrix(F,('w0','wa'),param_names)

In [10]:
param_labels[1:3],fiducial_fCPL,param_names

(['$w_0$', '$w_a$'], [0.315, -1.029, 0.21, -19.3], ['Om0', 'w0', 'wa', 'Mb'])

In [11]:
C_w0_wa

array([[ 0.00380852, -0.00894704],
       [-0.00894704,  0.12075617]])

In [12]:
fma.ellipse_parameters(C_w0_wa,0.32)

(0.5260591906987694,
 0.08442918693771034,
 94.34966006800019,
 0.13953305153130602)

In [13]:
# First plot in red
fig, axis_2d, axis_1d = fma.plot_fisher_matrices(F, fiducial_fCPL, param_names, 
                                             suptitle='Fisher Matrix Analysis - Mock NGRT data for kLCDM cosmology', 
                                             param_to_plot=['w0', 'wa'], param_labels=param_labels[1:3])

# Second plot in blue, using the same figure and axes
# fig, axis_2d, axis_1d = fma.plot_fisher_matrices(F_old, list(fiducial_kLCDM), param_names, 
#                                              suptitle='Fisher Matrix Analysis - Mock NGRT data for kLCDM cosmology', 
#                                              color='blue', param_to_plot=['Om0', 'Ode0'], param_labels=param_labels[0:2],
#                                              fig=fig, axs_2d=axis_2d, axs_1d=axis_1d)


# Manually add a custom legend for confidence levels and datasets
# legend_elements = [
#     Patch(facecolor='red', edgecolor='red', alpha=0.5, label=r'New Fisher Matrix (1$\sigma$, 2$\sigma$, 3$\sigma$)'),
#     Patch(facecolor='blue', edgecolor='blue', alpha=0.5, label=r'Old Fisher Matrix (1$\sigma$, 2$\sigma$, 3$\sigma$)')
# ]
# fig.legend(handles=legend_elements,bbox_to_anchor=(0.95, 0.8), fontsize=14)

# Show the final overlaid plot
plt.show()

AttributeError: module 'fisher_matrix_analysis' has no attribute 'plot_fisher_matrices'